# 04_event_cards

## Load

Load the event table and the evaluation set, and look at one event.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

EVENTS = Path("../data/events")
EVAL = Path("../evaluation/reference_events")

events = pd.read_parquet(EVENTS / "event_table.parquet")
evalset = pd.read_csv(EVAL / "evaluation_set.csv", index_col="event_id")

print("events:", events.shape, "| evaluation set:", evalset.shape)
print(events["cls"].value_counts().to_dict())
print()
print("columns:", list(events.columns))
print()
print(events.set_index("event_id").loc["E216"].to_string())


events: (302, 33) | evaluation set: (15, 19)
{'moderate': 171, 'quiet': 100, 'intense': 29, 'super': 2}

columns: ['type', 'cls', 'start', 'end', 'min_dst', 'min_time', 'core_start', 'core_end', 'event_id', 'overlaps_next', 'overlaps_prev', 'hours_start_to_min', 'recovery_hours', 'core_hours', 'bz_min', 'bz_min_time', 'hours_bz_le_-5', 'vbs_max', 'vbs_mean_12h_before_min', 'v_max', 'n_max', 'pdyn_max', 'bt_max', 'kp_max', 'ap_max', 'f107_mean', 'f107_spike_in_window', 'field_dscovr_pct', 'plasma_dscovr_pct', 'plasma_missing_pct', 'dst_min_is_final', 'kp_preliminary', 'marginal']

type                                     storm
cls                                      super
start                      2024-05-10 07:00:00
end                        2024-05-16 05:00:00
min_dst                                   -406
min_time                   2024-05-11 02:00:00
core_start                 2024-05-10 19:00:00
core_end                   2024-05-14 12:00:00
overlaps_next                        

## Hint rules

Define the hint rules (and check that each storm recovered), then test them on a few events.

In [2]:
aligned = pd.read_parquet(Path("../data/processed/aligned_hourly.parquet"))
events["dst_end"] = [aligned["dst"].loc[e] for e in events["end"]]
events["recovered"] = events["dst_end"] > -30
storm = events["type"] == "storm"
print("storms:", int(storm.sum()), "| not recovered above -30 nT by window end:", int((storm & ~events["recovered"]).sum()))
print(events.loc[storm & ~events["recovered"], ["event_id", "cls", "min_time", "min_dst", "dst_end", "recovery_hours"]].head(8).to_string(index=False))
print()

def hrs(n):
    return f"{int(n)} hour" + ("" if int(n) == 1 else "s")

def g_level(kp):
    """Approximate NOAA G-scale from the maximum Kp (>= 5, 6, 7, 8, 9)."""
    for lim, name in [(9, "G5"), (8, "G4"), (7, "G3"), (6, "G2"), (5, "G1")]:
        if kp >= lim - 0.01:
            return name
    return None

def make_hints(r):
    h = []
    if r["type"] == "quiet":
        h.append(f"Quiet interval: Dst stayed above -30 nT (minimum {r['min_dst']:.0f} nT).")
    else:
        h.append(f"{r['cls'].capitalize()} geomagnetic storm: Dst minimum {r['min_dst']:.0f} nT.")
        main = (r["min_time"] - r["core_start"]) / pd.Timedelta(hours=1)
        h.append(f"Dst first fell below -50 nT {hrs(main)} before reaching its minimum.")
        rec = r["recovery_hours"]
        if r["recovered"]:
            h.append(f"Dst was back above -30 nT {hrs(rec)} after the minimum.")
        else:
            h.append(f"Dst had not recovered above -30 nT by the end of the {hrs(rec)} after the minimum covered by this window.")

    g = g_level(r["kp_max"])
    h.append(f"Kp reached {r['kp_max']:.1f}" + (f" (approximately NOAA {g} level)." if g else " (below G1)."))

    n = int(r["hours_bz_le_-5"])
    if n >= 6:
        h.append(f"Sustained southward Bz: {hrs(n)} at or below -5 nT; minimum hourly Bz {r['bz_min']:.1f} nT.")
    elif n > 0:
        h.append(f"Brief southward Bz: {hrs(n)} at or below -5 nT; minimum hourly Bz {r['bz_min']:.1f} nT.")
    else:
        h.append(f"No southward Bz at or below -5 nT; minimum hourly Bz {r['bz_min']:.1f} nT.")
    if r["bz_min"] <= -20:
        h.append("Strongly southward Bz (hourly mean at or below -20 nT).")

    v = r["v_max"]
    if v >= 800:
        h.append(f"Very fast solar wind: maximum hourly speed {v:.0f} km/s.")
    elif v >= 600:
        h.append(f"Fast solar wind: maximum hourly speed {v:.0f} km/s.")
    else:
        h.append(f"Ordinary solar wind speed: maximum hourly speed {v:.0f} km/s.")

    if r["pdyn_max"] >= 10:
        h.append(f"High solar wind dynamic pressure: up to {r['pdyn_max']:.1f} nPa (protons only).")
    if r["vbs_max"] >= 5:
        h.append(f"Strong solar wind electric field: vBs up to {r['vbs_max']:.1f} mV/m.")

    f = r["f107_mean"]
    level = "low" if f < 100 else ("moderate" if f <= 150 else "high")
    h.append(f"Solar activity {level} (mean F10.7 {f:.0f} sfu).")
    return h

ev = events.set_index("event_id")
for eid in ["E216", "E143"]:
    print(eid, ev.loc[eid, "cls"])
    for line in make_hints(ev.loc[eid].to_dict()):
        print("  -", line)
    print()


storms: 202 | not recovered above -30 nT by window end: 0
Empty DataFrame
Columns: [event_id, cls, min_time, min_dst, dst_end, recovery_hours]
Index: []

E216 super
  - Super geomagnetic storm: Dst minimum -406 nT.
  - Dst first fell below -50 nT 7 hours before reaching its minimum.
  - Dst was back above -30 nT 123 hours after the minimum.
  - Kp reached 9.0 (approximately NOAA G5 level).
  - Sustained southward Bz: 23 hours at or below -5 nT; minimum hourly Bz -39.3 nT.
  - Strongly southward Bz (hourly mean at or below -20 nT).
  - Very fast solar wind: maximum hourly speed 1006 km/s.
  - High solar wind dynamic pressure: up to 42.0 nPa (protons only).
  - Strong solar wind electric field: vBs up to 27.0 mV/m.
  - Solar activity high (mean F10.7 222 sfu).

E143 quiet
  - Quiet interval: Dst stayed above -30 nT (minimum -6 nT).
  - Kp reached 3.0 (below G1).
  - Brief southward Bz: 1 hour at or below -5 nT; minimum hourly Bz -7.1 nT.
  - Ordinary solar wind speed: maximum hourly spee

## Event cards

Build the event card function and print one card as an example.

In [3]:
def iso(t):
    return None if pd.isna(t) else pd.Timestamp(t).strftime("%Y-%m-%dT%H:%MZ")

CLAIMS_NEED_SOURCE = [
    "physical cause of the event (for example a coronal mass ejection or a high-speed stream)",
    "impacts on satellites, GNSS, radio communication, power grids or aurora",
    "comparison with, or ranking against, other historical storms",
    "any forecast or prediction",
]

def make_card(r, role=None):
    storm_ev = r["type"] == "storm"
    caveats = [
        "All values are hourly means in UTC; minute-level extremes are larger.",
        "Solar wind and magnetic field values are shifted to the time they reach Earth (a delay of roughly 25 to 100 minutes depending on solar wind speed).",
        "Bz is in GSM coordinates; negative Bz means southward.",
    ]

    p = float(r["plasma_dscovr_pct"])
    if p < 50:
        caveats.append(f"Only {p:.0f}% of the solar wind plasma hours come from DSCOVR; the rest are OMNI hourly values because DSCOVR plasma was flagged as unreliable or missing.")
    elif p < 80:
        caveats.append(f"{p:.0f}% of the solar wind plasma hours come from DSCOVR; the rest are OMNI hourly values.")
    fl = float(r["field_dscovr_pct"])
    if fl < 90:
        caveats.append(f"Only {fl:.0f}% of the magnetic field hours come from DSCOVR; the rest are OMNI hourly values.")
    caveats.append("Dst minimum is final." if r["dst_min_is_final"] else "Dst is provisional and may be revised by Kyoto.")
    if r["overlaps_prev"]:
        caveats.append("This window overlaps the previous event.")
    if r["overlaps_next"]:
        caveats.append("This window overlaps the next event.")
    if r["dst_window_min"] < r["min_dst"]:
        caveats.append(f"The window also contains a deeper Dst value ({int(r['dst_window_min'])} nT) that belongs to the overlapping event.")
    if r["marginal"]:
        caveats.append("Brief event just beyond the -50 nT threshold.")
    if r["f107_spike_in_window"]:
        caveats.append("Daily F10.7 includes a flagged single-day spike.")
    if r["kp_preliminary"]:
        caveats.append("Kp includes preliminary values.")

    f = float(r["f107_mean"])
    return {
        "event_id": r["event_id"],
        "type": r["type"],
        "class": r["cls"],
        "evaluation_role": role,
        "window": {"start": iso(r["start"]), "end": iso(r["end"]),
                   "hours": int((r["end"] - r["start"]) / pd.Timedelta(hours=1)) + 1},
        "dst": {
            "min_nT": int(r["min_dst"]),
            "min_time": iso(r["min_time"]),
            "first_below_minus50_time": iso(r["core_start"]) if storm_ev else None,
            "hours_from_minimum_to_back_above_minus30": int(r["recovery_hours"]) if storm_ev else None,
            "minimum_is_final": bool(r["dst_min_is_final"]),
        },
        "drivers": {
            "bz_min_hourly_nT": round(float(r["bz_min"]), 1),
            "bz_min_time": iso(r["bz_min_time"]),
            "hours_bz_at_or_below_minus5": int(r["hours_bz_le_-5"]),
            "vbs_max_mV_per_m": round(float(r["vbs_max"]), 1),
            "vbs_mean_12h_before_dst_min_mV_per_m": round(float(r["vbs_mean_12h_before_min"]), 1),
            "speed_max_km_per_s": round(float(r["v_max"])),
            "density_max_per_cm3": round(float(r["n_max"]), 1),
            "dynamic_pressure_max_nPa_protons_only": round(float(r["pdyn_max"]), 1),
            "total_field_max_nT": round(float(r["bt_max"]), 1),
        },
        "geomagnetic": {"kp_max": round(float(r["kp_max"]), 2), "noaa_g_level_approx": g_level(float(r["kp_max"])),
                        "ap_max": int(r["ap_max"])},
        "solar": {"f107_mean_sfu": round(f), "level": "low" if f < 100 else ("moderate" if f <= 150 else "high")},
        "hints": make_hints(r),
        "data_quality": {"field_from_dscovr_pct": round(fl), "plasma_from_dscovr_pct": round(p),
                         "plasma_missing_pct": round(float(r["plasma_missing_pct"])), "caveats": caveats},
        "claims_requiring_a_retrieved_source": CLAIMS_NEED_SOURCE,
    }

events["dst_window_min"] = [aligned["dst"].loc[a:b].min() for a, b in zip(events["start"], events["end"])]
records = {r["event_id"]: r for r in events.to_dict("records")}
card = make_card(records["E216"], role="headline (super)")
print(json.dumps(card, indent=2))
print()
print("E143 caveats:", make_card(records["E143"])["data_quality"]["caveats"])

{
  "event_id": "E216",
  "type": "storm",
  "class": "super",
  "evaluation_role": "headline (super)",
  "window": {
    "start": "2024-05-10T07:00Z",
    "end": "2024-05-16T05:00Z",
    "hours": 143
  },
  "dst": {
    "min_nT": -406,
    "min_time": "2024-05-11T02:00Z",
    "first_below_minus50_time": "2024-05-10T19:00Z",
    "hours_from_minimum_to_back_above_minus30": 123,
    "minimum_is_final": false
  },
  "drivers": {
    "bz_min_hourly_nT": -39.3,
    "bz_min_time": "2024-05-10T21:00Z",
    "hours_bz_at_or_below_minus5": 23,
    "vbs_max_mV_per_m": 27.0,
    "vbs_mean_12h_before_dst_min_mV_per_m": 12.0,
    "speed_max_km_per_s": 1006,
    "density_max_per_cm3": 52.9,
    "dynamic_pressure_max_nPa_protons_only": 42.0,
    "total_field_max_nT": 71.6
  },
  "geomagnetic": {
    "kp_max": 9.0,
    "noaa_g_level_approx": "G5",
    "ap_max": 400
  },
  "solar": {
    "f107_mean_sfu": 222,
    "level": "high"
  },
  "hints": [
    "Super geomagnetic storm: Dst minimum -406 nT.",
    

## Build, validate and save

Build all 302 cards, check them against the aligned table, and save them.

In [4]:
roles = evalset["role"].to_dict()
cards = {eid: make_card(r, role=roles.get(eid)) for eid, r in records.items()}
print("cards:", len(cards), "| unique ids:", len(set(cards)))

# independent check against the aligned hourly table
aligned["vbs"] = aligned["v"] * (-aligned["bz"]).clip(lower=0) / 1000
bad = []
for eid, r in records.items():
    A = aligned.loc[r["start"]:r["end"]]
    c = cards[eid]
    own = aligned["dst"].loc[r["core_start"]:r["core_end"]].min() if r["type"] == "storm" else A["dst"].min()
    ok = (own == c["dst"]["min_nT"]
          and abs(A["kp"].max() - c["geomagnetic"]["kp_max"]) < 0.01
          and round(float(A["v"].max())) == c["drivers"]["speed_max_km_per_s"]
          and abs(A["bz"].min() - c["drivers"]["bz_min_hourly_nT"]) < 0.06
          and abs(A["vbs"].max() - c["drivers"]["vbs_max_mV_per_m"]) < 0.06)
    if not ok:
        bad.append(eid)
print("cards that disagree with the aligned table:", len(bad), bad[:5])
print("cards carrying the deeper-Dst caveat:", sum(any("deeper Dst" in x for x in c["data_quality"]["caveats"]) for c in cards.values()))

phrases = ["Sustained southward Bz", "Strongly southward Bz", "Very fast solar wind", "Fast solar wind",
           "High solar wind dynamic pressure", "Strong solar wind electric field"]
tab = pd.DataFrame({p: {k: np.mean([any(p in h for h in c["hints"]) for c in cards.values() if c["class"] == k]) * 100
                        for k in ["quiet", "moderate", "intense", "super"]} for p in phrases}).round(0)
print()
print("% of cards in each class carrying each hint:")
print(tab.to_string())

n_cav = pd.Series({k: np.mean([len(c["data_quality"]["caveats"]) for c in cards.values() if c["class"] == k])
                   for k in ["quiet", "moderate", "intense", "super"]}).round(1)
print()
print("mean number of caveats per card:", n_cav.to_dict())

with open(EVENTS / "event_cards.jsonl", "w", encoding="utf-8") as f:
    for c in cards.values():
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
(EVAL / "cards").mkdir(exist_ok=True)
for eid in evalset.index:
    with open(EVAL / "cards" / f"{eid}.json", "w", encoding="utf-8") as f:
        json.dump(cards[eid], f, indent=2, ensure_ascii=False)
print()
print("saved data/events/event_cards.jsonl and", len(evalset), "evaluation cards")

cards: 302 | unique ids: 302


cards that disagree with the aligned table: 0 []
cards carrying the deeper-Dst caveat: 2

% of cards in each class carrying each hint:
          Sustained southward Bz  Strongly southward Bz  Very fast solar wind  Fast solar wind  High solar wind dynamic pressure  Strong solar wind electric field
quiet                        2.0                    0.0                   0.0              3.0                               0.0                               0.0
moderate                    47.0                    1.0                   2.0             36.0                              18.0                              25.0
intense                    100.0                   24.0                  24.0             21.0                              69.0                              97.0
super                      100.0                  100.0                  50.0             50.0                             100.0                             100.0

mean number of caveats per card: {'quiet': 4.6, '


saved data/events/event_cards.jsonl and 15 evaluation cards


### 04 event cards findings

- Cards: `data/events/event_cards.jsonl` (302) and `evaluation/reference_events/cards/E###.json` (15). Each card has window, Dst, drivers, geomagnetic, solar, hints, data quality with caveats, and a list of claims that need a retrieved source. No causes or impacts appear in a card.
- Check: all 302 cards agree with an independent recomputation from the aligned table (storm Dst minimum taken over the event's own core). The check first flagged E159 and E282 because a neighbouring storm's deeper Dst value sits inside their window; both now carry a caveat.
- Hints separate the classes. Share of cards with each hint (quiet / moderate / intense / super): sustained southward Bz 2 / 47 / 100 / 100 %; strong electric field 0 / 25 / 97 / 100 %; high pressure 0 / 18 / 69 / 100 %; strongly southward Bz 0 / 1 / 24 / 100 %. Fast or very fast wind appears in 38% of moderate and 45% of intense storms, so speed does not separate those classes.
- Two quiet windows have sustained southward Bz despite Kp <= 3 and Dst > -30; useful as trap cases for the evaluation.
- Caveats per card: 4.6 quiet, 5.3 moderate, 5.2 intense, 6.5 super (three always-on caveats plus conditional ones).
- `evaluation_role` is bookkeeping only and is removed before cards go to the model.